# Regime Structure 01 — do Ivan's regimes exist as persistent states beyond volatility? (15 min – 4 h)

Pre-registration: `discussions/Regime_Structure/001_claude_branch_preregistration.md`. Definitions, null, windows and gates are fixed in
`src/data_engineering/modeling/regime_structure.py` (`registration()` below). No VWAP anywhere.

- **Data:** Binance USD-M perpetual 1m klines for BTC, ETH, SOL, XRP → 5m bars → clock-aligned windows of 15 min, 30 min, **1 h (primary)**, 2 h, 4 h ($n=3,6,12,24,48$ bars).
- **Hourly quantities:** $ER=\lvert\sum r\rvert/\sum\lvert r\rvert$, $s=\hat\sigma_{\text{pre}}\sqrt{12}$, shock $=\max\lvert r_5\rvert/\hat\sigma_{\text{pre}}$, time-of-day-relative volume and realised vol, window CVD change.
- **Labels (first match wins):** SHOCK ($\ge 3\hat\sigma$) → DISP (high $ER$, $\lvert\text{net}\rvert\ge s$, CVD agrees) → NOISE (low relative volume and vol) → RANGE (low $ER$, $\lvert\text{net}\rvert<s$) → OTHER. Thresholds = development-window quantiles, frozen.
- **Null (v1.1):** each 1m return and its taker delta multiplied by a random sign, the same sign for every asset in the same UTC minute — volatility, volume and time of day kept exactly, all directional structure destroyed.
- **Tests (test window from 2026-04-01, pooled):** lift $=P(Y_{h+1}\mid X_h)/P(Y_{h+1})$ for E1 RANGE→RANGE, E2 DISP→DISP, C1 RANGE→DISP, C2 DISP→NOISE, C3 up-DISP→SHOCK. Lifts pooled as the geometric mean over assets. Pass: lift / null median $\ge 1.10$ and permutation $p\le 0.05/5$ for the primary 1 h family, $p\le 0.05/20$ for the sweep family (15 min, 30 min, 2 h, 4 h). 399 null draws.

Environment: `RS01_ASSETS` (default BTC,ETH,SOL,XRP), `RS01_N_NULL` (default 399), `BINANCE_UM_DIR`, `RS01_SYNTHETIC=1` (synthetic smoke run, outputs to a temp dir).

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import regime_structure as rs

SYNTHETIC = os.getenv('RS01_SYNTHETIC', '').strip() == '1'
ASSETS = [a.strip().upper() for a in os.getenv('RS01_ASSETS', ','.join(rs.ASSETS)).split(',') if a.strip()]
P = replace(rs.Params(), n_null=int(os.getenv('RS01_N_NULL', str(rs.Params().n_null))))
if SYNTHETIC:
    BIN_DIR = None
    OUT = Path(tempfile.mkdtemp(prefix='rs01_'))
else:
    BIN_DIR = Path(os.getenv('BINANCE_UM_DIR') or Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Regime_Structure' / '01'
OUT.mkdir(parents=True, exist_ok=True)
REG = rs.registration(P)
print('Regime Structure 01 |', rs.VERSION, '| synthetic' if SYNTHETIC else f'| data {BIN_DIR}', '| out', OUT)
print(json.dumps({k: REG[k] for k in ('scales', 'shock_k', 'move_k', 'er_lo_q', 'er_hi_q', 'rvol_q', 'rrv_q', 'n_null',
                                       'alpha_per_test', 'alpha_sweep', 'min_effect', 'test_start')}, default=str))
SC = list(P.scales); N0 = P.n_bars
MIN = {n: f'{n * P.bar_min}m' if n * P.bar_min < 60 else f'{n * P.bar_min // 60}h' for n in SC}

## 1. Labels on real data and on the sign-randomised null

In [ ]:
RES = {}
for i, a in enumerate(ASSETS):
    t0 = time.time()
    if SYNTHETIC:
        df = rs.synthetic_klines(n_days=200, seed=20 + i, regime=(i % 2 == 0))
    else:
        path = BIN_DIR / f'{a}USDT_um_1m.parquet'
        if not path.exists():
            print(f'{a}: no Binance file at {path} -> skipped'); continue
        df = pd.read_parquet(path, columns=['open_time', 'close', 'volume', 'taker_buy_volume'])
    RES[a] = rs.run_asset(df, P)
    print(f'{a}: {RES[a]["minutes"]:,} minutes ({RES[a]["missing_minutes"]} missing), {len(RES[a]["labels"]):,} hours, '
          f'{P.n_null} null draws x {len(SC)} scales, {time.time() - t0:.0f}s')

base = pd.concat({(MIN[n], a): r['labels'][n].groupby('window')['label'].value_counts(normalize=True).unstack()
                  for n in SC for a, r in RES.items()}, names=['scale', 'asset']).reindex(columns=rs.LABELS).round(3)
print('Label base rates (share of windows):'); display(base)
thr = pd.DataFrame({(MIN[n], a): {k: v for k, v in r['thresholds'][n].items() if not isinstance(v, list)}
                    for n in SC for a, r in RES.items()}).T
print('Frozen thresholds (development window):'); display(thr.round(4))

## 2. Decisions — pooled over assets, test window (primary 1 h; sweep 15 min / 30 min / 2 h / 4 h)

In [ ]:
EVAL = []
for n in SC:
    ev = rs.evaluate([r['counts'][n] for r in RES.values()], [r['null_counts'][n] for r in RES.values()], P,
                     alpha=P.alpha if n == N0 else P.alpha_sweep)
    ev.insert(0, 'scale', MIN[n]); ev.insert(1, 'family', 'primary' if n == N0 else 'sweep')
    EVAL.append(ev)
allres = pd.concat(EVAL, ignore_index=True)
cols = ['scale', 'family', 'window', 'test', 'from', 'to', 'n_from', 'share_from_real', 'share_from_null',
        'p_to_given_from', 'p_to', 'lift_real', 'lift_null_median', 'lift_null_q95', 'ratio_to_null', 'p_value', 'alpha', 'pass']
display(allres[cols].round(4))
decision = allres[allres['window'] == 'test'][['scale', 'family', 'test', 'from', 'to', 'lift_real', 'ratio_to_null',
                                              'p_value', 'alpha', 'pass']]
print('DECISION (test window):'); display(decision.round(4))
print('Ratio to null by scale (test window):')
display(decision.pivot_table(index='test', columns='scale', values='ratio_to_null', sort=False)[[MIN[n] for n in SC]].round(3))

## 3. Descriptive — per asset, transitions, durations (no gates)

In [ ]:
per_asset = pd.concat([rs.evaluate([r['counts'][n]], [r['null_counts'][n]], P).assign(asset=a, scale=MIN[n])
                       for n in SC for a, r in RES.items()])
pa = per_asset[per_asset['window'] == 'test'].pivot_table(index=['scale', 'test'], columns='asset', values='ratio_to_null', sort=False)
print('Lift / null median, test window, per asset and scale:'); display(pa.round(3))

print('Primary scale (1 h) only below.')
tm_real = sum(rs.transition_matrix(r['labels'][N0], 'test') for r in RES.values()) / len(RES)
tm_null = sum(r['null_trans_mean'] for r in RES.values()) / len(RES)
print('Transition matrix P(next | current), test window, mean over assets — REAL:'); display(tm_real.round(3))
print('— NULL (mean over draws):'); display(tm_null.round(3))
print('REAL minus NULL:'); display((tm_real - tm_null).round(3))

sh = []
for a, r in RES.items():
    L = r['labels'][N0]; nxt = L.shift(-1)
    m = (L['label'] == 'DISP') & L['up'] & (nxt['label'] == 'SHOCK') & (nxt['window'] == 'test')
    allsh = (L['label'] == 'SHOCK') & (L['window'] == 'test')
    sh.append({'asset': a, 'upDISP_then_SHOCK': int(m.sum()),
               'share_shock_up_after_upDISP': float(nxt.loc[m, 'shock_up'].mean()) if m.any() else np.nan,
               'share_shock_up_all_shocks': float(L.loc[allsh, 'shock_up'].mean())})
shock_dir = pd.DataFrame(sh)
print('Direction of the largest 5m bar in SHOCK hours that follow an up-displacement (test window):'); display(shock_dir.round(3))

runs = pd.DataFrame({'real_mean_hours': sum(rs.run_lengths(r['labels'][N0], 'test') for r in RES.values()) / len(RES),
                     'null_mean_hours': sum(r['null_runs_mean'] for r in RES.values()) / len(RES)})
print('Mean run length (hours), test window:'); display(runs.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
tests = [t for t, _, _ in rs.TESTS]
for k, t in enumerate(tests):
    NB = [r['null_counts'][N0] for r in RES.values()]; RB = [r['counts'][N0] for r in RES.values()]
    ln = np.array([rs.pooled_lift([nb[j] for nb in NB], ('test', t), null=True) for j in range(P.n_null)])
    ax.scatter(np.full(len(ln), k) + np.random.default_rng(k).uniform(-0.15, 0.15, len(ln)), ln, s=6, alpha=0.4, color='grey')
    ax.scatter([k], [rs.pooled_lift(RB, ('test', t))], s=80, color='crimson', zorder=3)
ax.axhline(1, color='black', lw=0.7)
ax.set_xticks(range(len(tests)), [f'{t}\n{x}→{y}' for t, x, y in rs.TESTS])
ax.set_ylabel('lift  P(Y next | X) / P(Y)'); ax.set_title('1 h, test window: real (red) vs sign-randomised null (grey)')
plt.tight_layout(); plt.show()

## 4. Save compact outputs

In [ ]:
base.reset_index().to_csv(OUT / '01_label_base_rates.csv', index=False)
thr.to_csv(OUT / '01_thresholds.csv')
allres.to_csv(OUT / '01_results_all_scales.csv', index=False)
decision.to_csv(OUT / '01_decision.csv', index=False)
per_asset.to_csv(OUT / '01_per_asset.csv', index=False)
tm_real.to_csv(OUT / '01_transitions_real_test.csv'); tm_null.to_csv(OUT / '01_transitions_null_test.csv')
runs.to_csv(OUT / '01_run_lengths_test.csv')
shock_dir.to_csv(OUT / '01_shock_direction_test.csv', index=False)
manifest = {'registration': REG, 'assets_run': list(RES), 'synthetic': SYNTHETIC,
            'decision': decision.to_dict(orient='records'), 'run_utc': str(pd.Timestamp.now(tz='UTC'))}
(OUT / '01_manifest.json').write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
print('saved to', OUT, sorted(p.name for p in OUT.iterdir()))